# 🔄 基本代理工作流程與 Azure OpenAI（Responses API）（.NET）

## 📋 工作流程編排教學

本筆記本示範如何使用 Microsoft Agent Framework for .NET 與 Azure OpenAI（Responses API）構建複雜的<strong>代理工作流程</strong>。你將學習如何創建多步驟的業務流程，其中 AI 代理通過結構化的編排模式協同完成複雜任務。

## 🎯 學習目標

### 🏗️ <strong>工作流程架構基礎</strong>
- **Workflow Builder**：設計和編排複雜的多步驟 AI 流程
- **Agent Coordination**：在工作流程中協調多個專門代理
- **Azure OpenAI (Responses API)**：在工作流程中利用 Azure OpenAI Responses API
- **Visual Workflow Design**：創建並視覺化工作流程結構以增進理解

### 🔄 <strong>流程編排模式</strong>
- **Sequential Processing**：以邏輯順序串聯多個代理任務
- **State Management**：在工作流程階段間維護上下文和數據流
- **Error Handling**：實施穩健的錯誤恢復和工作流程彈性
- **Performance Optimization**：設計高效工作流程以支援企業級運營

### 🏢 <strong>企業工作流程應用</strong>
- **Business Process Automation**：自動化複雜的組織流程
- **Content Production Pipeline**：包含審核和批准階段的編輯工作流程
- **Customer Service Automation**：多步驟的客戶查詢解決流程
- **Data Processing Workflows**：使用 AI 驅動轉換的 ETL 工作流程

## ⚙️ 前置條件與設定

### 📦 **必需的 NuGet 套件**

本工作流程示範使用多個核心 .NET 套件：

```xml
<!-- Core AI Framework -->
<PackageReference Include="Microsoft.Extensions.AI" Version="9.9.0" />

<!-- Azure OpenAI (Responses API) -->
<PackageReference Include="Azure.AI.OpenAI" Version="2.1.0" />
<PackageReference Include="Azure.Identity" Version="1.13.1" />

<!-- Agent Framework (Local Development) -->
<!-- Microsoft.Agents.AI.dll - Core agent abstractions -->
<!-- Microsoft.Agents.AI.OpenAI.dll - Azure OpenAI (Responses API) integration -->

<!-- Configuration and Environment -->
<PackageReference Include="DotNetEnv" Version="3.1.1" />
```

### 🔑 **Azure OpenAI 配置**

**環境設定 (.env 檔案)：**
```env
AZURE_OPENAI_ENDPOINT=https://<your-resource>.openai.azure.com
AZURE_OPENAI_DEPLOYMENT=gpt-5-mini
```

**Azure OpenAI 存取：**
1. 在 Azure 入口網站建立 Azure OpenAI 資源
2. 部署一個模型（例如，`gpt-5-mini`）並記錄部署名稱
3. 使用 `az login` 登入並按上述示範設定環境變數

### 🏗️ <strong>工作流程架構概述</strong>

```mermaid
graph TD
    A[工作流程建構器] --> B[代理註冊表]
    B --> C[工作流程執行引擎]
    C --> D[Agent 1: 內容產生器]
    C --> E[Agent 2: 內容審核者] 
    D --> F[工作流程結果]
    E --> F
    G[Azure OpenAI（回應 API）] --> D
    G --> E
```

**關鍵元件：**
- **WorkflowBuilder**：設計工作流程的主要編排引擎
- **AIAgent**：具有特定能力的個別專門代理
- **Azure OpenAI Client**：Azure OpenAI Responses API 整合
- **Execution Context**：管理工作流程各階段間的狀態與數據流

## 🎨 <strong>企業工作流程設計模式</strong>

### 📝 <strong>內容生產工作流程</strong>
```
User Request → Content Generation → Quality Review → Final Output
```

### 🔍 <strong>文件處理管線</strong>
```
Document Input → Analysis → Extraction → Validation → Structured Output
```

### 💼 <strong>商業情報工作流程</strong>
```
Data Collection → Processing → Analysis → Report Generation → Distribution
```

### 🤝 <strong>客戶服務自動化</strong>
```
Customer Inquiry → Classification → Processing → Response Generation → Follow-up
```

## 🏢 <strong>企業效益</strong>

### 🎯 <strong>可靠性與可擴展性</strong>
- **Deterministic Execution**：一致且可重複的工作流程結果
- **Error Recovery**：優雅處理各階段失敗
- **Performance Monitoring**：追蹤執行指標與優化機會
- **Resource Management**：有效分配及使用 AI 模型資源

### 🔒 <strong>安全性與合規性</strong>
- **Secure Authentication**：透過 `az login`（AzureCliCredential）進行 Microsoft Entra ID 驗證
- **Audit Trails**：完整紀錄工作流程執行及決策點
- **Access Control**：對工作流程執行及監控施行細緻權限
- **Data Privacy**：在整個工作流程中安全處理敏感資訊

### 📊 <strong>可觀察性與管理</strong>
- **Visual Workflow Design**：清晰呈現流程流向與相依關係
- **Execution Monitoring**：即時追蹤工作流程進度與效能
- **Error Reporting**：詳盡錯誤分析與除錯能力
- **Performance Analytics**：優化及容量規劃的衡量指標

讓我們開始構建你的第一個企業級 AI 工作流程吧！🚀


In [1]:
#r "nuget: Microsoft.Extensions.AI, 9.9.1"

Installed Packages Microsoft.Extensions.AI, 9.9.1

In [ ]:
#r "nuget: Azure.AI.OpenAI, 2.1.0"

Installed Packages System.ClientModel, 1.6.1

In [3]:
#r "nuget: Azure.Identity, 1.15.0"
#r "nuget: System.Linq.Async, 6.0.3"
#r "nuget: OpenTelemetry.Api, 1.0.0"
#r "nuget: OpenTelemetry.Api, 1.0.0"

Installed Packages Azure.Identity, 1.15.0 OpenTelemetry.Api, 1.0.1 System.Linq.Async, 6.0.3

In [5]:

#r "nuget: Microsoft.Agents.AI.Workflows, 1.0.0-preview.251001.3"

Installed Packages Microsoft.Agents.AI.Workflows, 1.0.0-preview.251001.3

In [ ]:

#r "nuget: Microsoft.Agents.AI.OpenAI, 1.0.0-preview.251001.3"

Installed Packages Microsoft.Agents.AI.OpenAI, 1.0.0-preview.251001.2

In [7]:
#r "nuget: DotNetEnv, 3.1.1"

Installed Packages DotNetEnv, 3.1.1

In [8]:
// #r "nuget: Microsoft.Extensions.AI.OpenAI, 9.9.0-preview.1.25458.4"

In [ ]:
using System;
using System.ComponentModel;
using Azure.AI.OpenAI;
using Azure.Identity;
using Microsoft.Extensions.AI;
using Microsoft.Agents.AI;
using Microsoft.Agents.AI.Workflows;

In [10]:
 using DotNetEnv;

In [11]:
Env.Load("../../../.env");

In [ ]:
// Azure OpenAI with the Responses API (stable v1 endpoint). Sign in with `az login`.
var azureEndpoint = Environment.GetEnvironmentVariable("AZURE_OPENAI_ENDPOINT") ?? throw new InvalidOperationException("AZURE_OPENAI_ENDPOINT is not set.");
var deployment = Environment.GetEnvironmentVariable("AZURE_OPENAI_DEPLOYMENT") ?? "gpt-5-mini";

In [ ]:
// The Azure OpenAI client is created directly from the endpoint and Azure CLI credential — no custom client options are required.

In [ ]:
var azureClient = new AzureOpenAIClient(new Uri(azureEndpoint), new AzureCliCredential());

In [15]:
const string ReviewerAgentName = "Concierge";
const string ReviewerAgentInstructions = @"
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. ";

In [16]:
const string FrontDeskAgentName = "FrontDesk";
const string FrontDeskAgentInstructions = @"""
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """;

In [ ]:
AIAgent reviewerAgent = azureClient.GetOpenAIResponseClient(deployment).CreateAIAgent(
    name:ReviewerAgentName,instructions:ReviewerAgentInstructions);
AIAgent frontDeskAgent  = azureClient.GetOpenAIResponseClient(deployment).CreateAIAgent(
    name:FrontDeskAgentName,instructions:FrontDeskAgentInstructions);

In [18]:
var workflow = new WorkflowBuilder(frontDeskAgent)
            .AddEdge(frontDeskAgent, reviewerAgent)
            .Build();

In [19]:
ChatMessage userMessage = new ChatMessage(ChatRole.User, [
	new TextContent("I would like to go to Paris.") 
]);

In [20]:
StreamingRun run = await InProcessExecution.StreamAsync(workflow, userMessage);

In [21]:
await run.TrySendMessageAsync(new TurnToken(emitEvents: true));
string id="";
string messageData="";
await foreach (WorkflowEvent evt in run.WatchStreamAsync().ConfigureAwait(false))
{
    if (evt is AgentRunUpdateEvent executorComplete)
    {
        if(id=="")
        {
            id=executorComplete.ExecutorId;
        }
        if(id==executorComplete.ExecutorId)
        {
            messageData+=executorComplete.Data.ToString();
        }
        else
        {
            id=executorComplete.ExecutorId;
        }
        // Console.WriteLine($"{executorComplete.ExecutorId}: {executorComplete.Data}");
    }
}

Console.WriteLine(messageData);

Visit the Louvre Museum. It's a must-see for art enthusiasts and history lovers.That recommendation is quite popular and likely to attract many tourists. To refine it for a more local and authentic experience, consider suggesting an alternative that focuses on smaller, lesser-known art venues or galleries. Look for places where local artists exhibit or community spaces that host cultural events. This approach allows travelers to connect with the local art scene more intimately, away from the typical tourist routes.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
